# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer Exploration with `mlcroissant`
This notebook provides a walkthrough for loading and exploring the FAIR^2 dataset using the `mlcroissant` library.

### Dataset Source
The dataset is described with a Croissant schema and hosted at the following URL:

`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`


In [ ]:
# Ensure mlcroissant is installed
!pip install --quiet mlcroissant

## 1. Data Loading
Load the dataset metadata and records from the Croissant schema using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Access the dataset metadata object
metadata = dataset.metadata

# Print high-level summary
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Let's review the available record sets and their fields, along with their `@id` values for referencing.

We'll enumerate the available record sets and list their fields' `@id` and human-readable names.

In [ ]:
# List all available record sets, their @id, name, and field @id
print("Available record sets in the dataset:")
record_sets = []
if hasattr(metadata, 'record_sets'):
    for rs in metadata.record_sets:
        print(f"  - Record set name: {getattr(rs, 'name', '[No name]')}")
        print(f"    @id: {rs.id}")
        if hasattr(rs, 'fields'):
            print(f"    Fields:")
            for field in rs.fields:
                fname = getattr(field, 'name', '[No name]')
                print(f"      - {fname} (@id: {field.id})")
        else:
            print("    [No fields listed]")
        record_sets.append(rs.id)
else:
    print("No record sets found in schema.")

## 3. Data Extraction
We'll load data from each record set into separate pandas DataFrames for exploration.

We use the record set and field `@id`s from the above overview.

In [ ]:
dataframes = {}

for record_set_id in record_sets:
    print(f"Loading data from record set {record_set_id}...")
    # Read all records for this record set
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"\nColumns for {record_set_id}: {list(df.columns)}\n---")

# For demonstration, select the first record set loaded for detailed exploration
main_record_set_id = record_sets[0] if record_sets else None
if main_record_set_id and main_record_set_id in dataframes:
    print(f"\nPreview of {main_record_set_id} DataFrame:")
    display(dataframes[main_record_set_id].head())
else:
    print("No record sets loaded.")

## 4. Exploratory Data Analysis (EDA)
Let's perform some basic EDA:
- Filter records based on a numeric field (e.g., age or diagnosis interval if such a field exists)
- Normalize the numeric field
- Group and summarize by a categorical field (such as sex or cancer site)

Please customize the `numeric_field_id` and `group_field_id` as appropriate, referencing the `@id`s from above.

In [ ]:
# Select the main DataFrame if available
df = dataframes.get(main_record_set_id)

# For reproducibility, set field IDs for numeric and categorical fields.
# Replace these with actual available @id values if necessary.

# Example field ids based on common variable names; please change if different in this dataset
numeric_field_id = None
group_field_id = None

# Suggesting likely field ids by scanning columns for typical field names
if df is not None:
    lower_columns = [c.lower() for c in df.columns]
    possible_numeric_fields = [col for col in df.columns if 'age' in col.lower() or 'interval' in col.lower()]
    possible_group_fields = [col for col in df.columns if 'sex' in col.lower() or 'site' in col.lower() or 'msi' in col.lower() or 'comorbid' in col.lower()]
    if possible_numeric_fields:
        numeric_field_id = possible_numeric_fields[0]  # Choose the first likely numeric field
    if possible_group_fields:
        group_field_id = possible_group_fields[0]  # Choose the first likely categorical field

if not numeric_field_id:
    # Fallback: just use the first column
    numeric_field_id = df.columns[0] if df is not None and len(df.columns) else None

if not group_field_id:
    # Fallback: use the second column
    group_field_id = df.columns[1] if df is not None and len(df.columns) > 1 else None

# EDA: filter, normalize, and group on the found fields
if df is not None and numeric_field_id in df.columns:
    print(f"Using numeric field '{numeric_field_id}' and group field '{group_field_id}'")
    # Try convert to numeric (non-numeric become NaN)
    df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
    threshold = df[numeric_field_id].mean() if df[numeric_field_id].notnull().any() else 0
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records where {numeric_field_id} > mean ({threshold:.2f}): {len(filtered_df)} rows")
    display(filtered_df.head())

    # Normalize the numeric field
    norm_col = f"{numeric_field_id}_normalized"
    filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"\nNormalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, norm_col]].head())

    # Grouping if group field is available and not null
    if group_field_id and group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"\nGrouped mean of {numeric_field_id} by {group_field_id}:")
        display(grouped_df)
    else:
        print(f"\nGroup field '{group_field_id}' is not in columns.")
else:
    print("No valid numeric field or data available for EDA.")

## 5. Visualization
We visualize the distribution of the chosen numeric field and the group-wise means if available.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if df is not None and numeric_field_id in df.columns:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field_id].dropna(), bins=15, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()

    if group_field_id and group_field_id in df.columns:
        plt.figure(figsize=(8,4))
        sns.boxplot(data=df, x=group_field_id, y=numeric_field_id)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xticks(rotation=30)
        plt.show()

## 6. Conclusion
- This notebook demonstrated how to load and explore the FAIR^2 dataset using the `mlcroissant` library.
- We inspected available record sets and fields (using `@id` references), loaded tabular data, and performed basic filtering, normalization, grouping, and visualization.
- This workflow supports further clinical or methodological analyses on second primary colorectal cancer phenotypes in cancer survivors.